In [1]:
import numpy as np
import tensorflow as tf
from keras import layers, models
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import train_test_split
import os
from keras.callbacks import EarlyStopping
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc
from sklearn.preprocessing import label_binarize

In [5]:
# ============================================================
# 1. CARGA EFICIENTE DEL DATASET MEDIANTE MEMORIA MAPEADA
# ============================================================

from pathlib import Path
import numpy as np

RUTA_DATOS = Path("Data") / "Raw_data"

ruta_X = RUTA_DATOS / "X_data_float32.npy"
ruta_Y = RUTA_DATOS / "Y_data_int16.npy"

if not ruta_X.exists():
    raise FileNotFoundError(
        f"No se encontró {ruta_X}\n"
        "Primero convierte el dataset original a archivos .npy."
    )

if not ruta_Y.exists():
    raise FileNotFoundError(
        f"No se encontró {ruta_Y}\n"
        "Primero convierte el dataset original a archivos .npy."
    )

# mmap_mode evita cargar todo el archivo en la RAM.
X_data = np.load(ruta_X, mmap_mode="r")
Y_data = np.load(ruta_Y, mmap_mode="r")

print("Dataset cargado mediante memoria mapeada")
print(f"Forma de X: {X_data.shape}")
print(f"Tipo de X:  {X_data.dtype}")
print(f"Forma de Y: {Y_data.shape}")
print(f"Tipo de Y:  {Y_data.dtype}")

tamano_X_gib = X_data.size * X_data.dtype.itemsize / 1024**3
tamano_Y_mib = Y_data.size * Y_data.dtype.itemsize / 1024**2

print(f"Tamaño de X en disco: {tamano_X_gib:.2f} GiB")
print(f"Tamaño de Y en disco: {tamano_Y_mib:.2f} MiB")

FileNotFoundError: No se encontró Data\Raw_data\X_data_float32.npy
Primero convierte el dataset original a archivos .npy.

In [ ]:
# 2. PREPARAR ETIQUETAS Y TENSORES
# Convertir letras (N, V, R...) a números enteros (0 a 7) para la red neuronal
codificador = LabelEncoder()
Y_numerico = codificador.fit_transform(Y_filtrado)

# Asegurar que el tensor X tenga la dimensión de canal (profundidad = 1 para CNN)
X_cnn = np.expand_dims(X_filtrado, axis=-1) # Pasa de (N, 6, 54) a (N, 6, 54, 1)

# Dividir en datos de Entrenamiento (80%) y Validación (20%)
X_train, X_val, y_train, y_val = train_test_split(X_cnn, Y_numerico, test_size=0.2, random_state=42)

In [ ]:
# 3. CALCULAR LOS PESOS DE CLASE (El secreto del balanceo)
# Esto penaliza matemáticamente los errores en las clases más pequeñas (ej. F, f, A)
pesos = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
# Convertir el arreglo a un diccionario que Keras pueda entender
diccionario_pesos = dict(enumerate(pesos))
print("\nPesos asignados por clase (La clase N tendrá un peso muy bajo, y la F muy alto):")
for clase, peso in zip(codificador.classes_, pesos):
    print(f"Clase '{clase}': {peso:.4f}")

In [ ]:
# 4. ARQUITECTURA TRANSFER LEARNING
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input

backbone=EfficientNetB0(include_top=False,weights="imagenet",input_shape=(128,128,3))
backbone.trainable=False

inp=layers.Input(shape=(29,112,1))
x=layers.Resizing(128,128)(inp)
x=layers.Concatenate()([x,x,x])
x=preprocess_input(x)
x=backbone(x,training=False)
x=layers.Conv2D(256,3,padding="same",activation="relu")(x)
x=layers.BatchNormalization()(x)
x=layers.GlobalAveragePooling2D()(x)
x=layers.Dense(256,activation="relu")(x)
x=layers.Dropout(0.5)(x)
x=layers.Dense(128,activation="relu")(x)
x=layers.Dropout(0.3)(x)
out=layers.Dense(len(clases_viables)+1,activation="softmax")(x)
modelo=tf.keras.Model(inp,out)
modelo.summary()


In [ ]:
# 5. COMPILAR Y ENTRENAR
from keras.callbacks import EarlyStopping,ReduceLROnPlateau,ModelCheckpoint
modelo.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
loss="sparse_categorical_crossentropy",
metrics=["accuracy",tf.keras.metrics.Precision(),tf.keras.metrics.Recall()])
callbacks=[
EarlyStopping(monitor="val_loss",patience=6,restore_best_weights=True),
ReduceLROnPlateau(monitor="val_loss",factor=0.2,patience=3,min_lr=1e-6),
ModelCheckpoint("mejor_modelo.keras",save_best_only=True,monitor="val_accuracy")]
historial=modelo.fit(X_train,y_train,validation_data=(X_val,y_val),epochs=40,batch_size=32,class_weight=diccionario_pesos,callbacks=callbacks)

backbone.trainable=True
for l in backbone.layers[:-40]:
    l.trainable=False
modelo.compile(optimizer=tf.keras.optimizers.Adam(1e-5),loss="sparse_categorical_crossentropy",metrics=["accuracy"])
historial_ft=modelo.fit(X_train,y_train,validation_data=(X_val,y_val),epochs=10,batch_size=32,class_weight=diccionario_pesos,callbacks=callbacks)


In [ ]:
# 1. HACER PREDICCIONES CON EL SET DE VALIDACIÓN
#modelo=load_model(r'Data\Models\modelo_arritmias_cnn_1.0.keras')
# Obtenemos las probabilidades de la red
y_pred_prob = modelo.predict(X_val)
# Nos quedamos con la clase que tenga mayor probabilidad
y_pred_clases = np.argmax(y_pred_prob, axis=1)

# Nombres de las clases (recuperados de tu codificador anterior)
etiquetas = codificador.classes_

# 2. MATRIZ DE CONFUSIÓN
matriz = confusion_matrix(y_val, y_pred_clases)

plt.figure(figsize=(10, 8))
sns.heatmap(matriz, annot=True, fmt='d', cmap='Blues', 
            xticklabels=etiquetas, yticklabels=etiquetas)
plt.title('Matriz de Confusión: ¿Qué está confundiendo la red?')
plt.ylabel('Latido Real (Verdad)')
plt.xlabel('Predicción del Modelo')
plt.show()

# 3. REPORTE DE CLASIFICACIÓN (Métricas duras)
print("\n=== REPORTE DE CLASIFICACIÓN (PRECISIÓN POR CLASE) ===")
print(classification_report(y_val, y_pred_clases, target_names=etiquetas))

# 4. CURVA ROC MULTICLASE
# Necesitamos binarizar las etiquetas para hacer ROC por cada clase
y_val_bin = label_binarize(y_val, classes=np.arange(len(etiquetas)))

plt.figure(figsize=(10, 8))
for i in range(len(etiquetas)):
    # Calcular Tasa de Falsos Positivos y Verdaderos Positivos
    fpr, tpr, _ = roc_curve(y_val_bin[:, i], y_pred_prob[:, i])
    roc_auc = auc(fpr, tpr) # Área bajo la curva
    
    plt.plot(fpr, tpr, lw=2, label=f'Clase {etiquetas[i]} (AUC = {roc_auc:.2f})')

# Línea diagonal (el modelo adivina al azar)
plt.plot([0, 1], [0, 1], 'k--', lw=2)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('Tasa de Falsos Positivos (FPR)')
plt.ylabel('Tasa de Verdaderos Positivos (TPR)')
plt.title('Curva ROC: Capacidad de distinción por arritmia')
plt.legend(loc="lower right")
plt.show()

In [ ]:
archivo_modelo=r"Data\Models\modelo_arritmias_cnn_4.0"

# Guardar el modelo físico en tu disco duro
print(f"\nGuardando el modelo definitivo en: {archivo_modelo}")
modelo.save(archivo_modelo)
print("-" * 50)
print("¡Modelo entrenado y guardado con éxito! Listo para producción.")
